# 1. Initial Analysis

In this notebook we explore the raw data file `data/all_car_adverts.csv`. We make some obvious observations which answer some of the following questions:

- What is this dataset for?
- What columns are in the dataset?

We also state any initial observations which may prove useful in Data Wrangling.

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/all_car_adverts.csv")
df.head()

,Unnamed: 0,make,model,variant,car_price,car_badges,car_title,car_sub_title,car_attention_grabber,car_specs,...,num_owner,ulez,full_service,part_service,part_warranty,full_dealership,first_year_road_tax,brand_new,finance_available,discounted
0,0,AC,Cobra,NaN,89995.0,NaN,AC Cobra,4.9 MK IV CRS 2dr,GENUINE AC COBRA CRS 522 BHP,"2001 (X reg), Convertible, 14,400 miles, 4.9L,...",...,5.0,0,0,0,0,0,0,0,0,0
1,1,AC,Cobra,NaN,92500.0,'',AC Cobra,378 - MkIV,PHYSICAL CAR!,"2019 (T reg), Convertible, 650 miles, Manual, ...",...,NaN,0,0,0,0,0,0,0,0,0
2,2,AC,Cobra,NaN,109995.0,'',AC Cobra,MK1V 212 SC. 3.5 V8 350 BHP LOTUS TWIN TURBO. ...,FULL CARBON FIBRE BODY.,"2000 (X reg), Convertible, 21,600 miles, 3.5L,...",...,3.0,0,0,0,0,0,0,0,0,0
3,3,AC,Cobra,NaN,124950.0,'',AC Cobra,302 MKIV 2dr,ABSOLUTELY STUNNING,"1989 (F reg), Convertible, 2,750 miles, Manual...",...,NaN,0,0,0,0,0,0,0,0,0
4,4,AC,Cobra,NaN,124950.0,'',AC Cobra,302 MKIV With Factory Lightweight Engine 5.0 2dr,'STAGE 3' SVO ENGINE,"1989 (E reg), Convertible, 15,142 miles, 5.0L,...",...,NaN,0,0,0,0,0,0,0,0,0


In [3]:
df.shape

(818456, 32)

## 1.1. What is this dataset?
According to the [dataset description on Kaggle](https://www.kaggle.com/datasets/guanhaopeng/uk-used-car-market), the dataset is made up of scraped data from a used car website in October 2022. It consists of ~800k records with 32 variables. We note that the exact origin of the dataset is unknown and undocumented. However, for the sake of this project, we are not too concerned with the data being "real".

## 1.2. What columns are in the dataset?

In [4]:
df.columns.values

<StringArray>
[           'Unnamed: 0',                  'make',                 'model',
               'variant',             'car_price',            'car_badges',
             'car_title',         'car_sub_title', 'car_attention_grabber',
             'car_specs',            'car_seller',     'car_seller_rating',
   'car_seller_location',                  'year',                   'reg',
             'body_type',                 'miles',            'engine_vol',
           'engine_size',      'engine_size_unit',          'transmission',
             'feul_type',             'num_owner',                  'ulez',
          'full_service',          'part_service',         'part_warranty',
       'full_dealership',   'first_year_road_tax',             'brand_new',
     'finance_available',            'discounted']
Length: 32, dtype: str

Each record represents an individual advert and contains information about the advert itself, e.g. `car_attention_grabber` but also details of the car itself e.g. `transmission` and `miles`. The purpose of this project is to use details about the car in order to predict `car_price`. Thus, it's likely that not all columns will be necessary for our purposes.

We also note that these columns are not all complete. For example, `num_owners`, which one would assume would be important for prediction, is not complete for some records. Further, some columns are ambiguous and not properly documented, e.g. `full_dealership` and `full_service`.

In [7]:
df[df["full_service"] > 1].head()

,Unnamed: 0,make,model,variant,car_price,car_badges,car_title,car_sub_title,car_attention_grabber,car_specs,...,num_owner,ulez,full_service,part_service,part_warranty,full_dealership,first_year_road_tax,brand_new,finance_available,discounted


`full_service` is a boolean column which, assumably, represents whether or not a car has received a full service before being posted on the site.

In [10]:
df[df["full_dealership"] > 1].head()

,Unnamed: 0,make,model,variant,car_price,car_badges,car_title,car_sub_title,car_attention_grabber,car_specs,...,num_owner,ulez,full_service,part_service,part_warranty,full_dealership,first_year_road_tax,brand_new,finance_available,discounted


`full_dealership` is also a boolean column. The exact meaning of this column is ambiguous and not stated in the description. It _could_ relate to warranty, as there are other warranty columns in the dataset. However, it could also refer to the fact that all previous services have been conducted by a dealership (of the appropriate brand). The latter would make more sense since the warranty columns refer to warranty in the column name.

In [16]:
df[
    (df["full_dealership"] == 1)
    & (df["full_service"] == 1)
].head()

,Unnamed: 0,make,model,variant,car_price,car_badges,car_title,car_sub_title,car_attention_grabber,car_specs,...,num_owner,ulez,full_service,part_service,part_warranty,full_dealership,first_year_road_tax,brand_new,finance_available,discounted


However, the fact that there are no cars in the dataset which have both

- Received a full service
- Have received all their services from the dealership of the car's brand

makes me doubt the latter case also.

We note that `car_specs` is a string description of the car specifications. Most of which are specified from the other columns. We look at the first record as an example.

In [17]:
record1 = df.iloc[0]
record1

Unnamed: 0                                                               0
make                                                                    AC
model                                                                Cobra
variant                                                                NaN
car_price                                                          89995.0
car_badges                                                             NaN
car_title                                                         AC Cobra
car_sub_title                                            4.9 MK IV CRS 2dr
car_attention_grabber                         GENUINE AC COBRA CRS 522 BHP
car_specs                2001 (X reg), Convertible, 14,400 miles, 4.9L,...
car_seller                                                  Private seller
car_seller_rating                                                      NaN
car_seller_location                                                watford
year                     

The car's specification string states

> 2001 (X reg), Convertible, 14,400 miles, 4.9L, 225BHP, Manual, Petrol, 5 owners

and we can see from the other columns:

- `year` = 2001
- `reg` = X reg
- `miles` = 14400
- `engine_vol` = 4.9
- `engine_size` = 225
- `engine_size_unit` = BHP
- `transmission` = Manual
- `feul_type` = Petrol
- `num_owner` = 5.0

We of course note some of the glaringly obvious things that we will have to do at some stage of the project

- Categorise some of the string columns, e.g. `[Manual, Automatic]` -> `[0, 1]`
- Fix some of the typos, e.g. `feul_type` -> `fuel_type`
- Convert `engine_size` to the same units.

Moreover, we note that the `reg` column doesn't give us the full registration string of the car. It only gives us a single character. In the UK, following 2001, the letter denotes the area in which the car was registered. Of course, this means we have to check if there are cars older than September 2001.

In [24]:
df["year"].sort_values()

186147    101 miles
586693         1079
5283           1934
5284           1934
612050         1934
            ...    
106924       Saloon
35158        Saloon
102059       Saloon
102053       Saloon
449519       Saloon
Name: year, Length: 818456, dtype: str

We've discovered a terrible problem: the `year` column contains data which corresponds to different columns, e.g. milage, body type. Of course `1079` is _not_ the year the car was registered, but it's also not obvious which column it should belong to.

## 1.3. Summary

- The dataset contains data about listings on a used car sales website.
- It contains data about the listing itself as well as details about the car.
- The exact origins of the dataset is unknown, but in this project we aren't too concerned with this fact.
- We don't know what some of the columns mean due to the lack of documentation and context (e.g. `full_dealership`).
- The dataset is extremely messy, and some data is written in columns where it should not be written (e.g. the `year` column).